In [9]:
import pandas as pd
import re, unicodedata

In [2]:
def clean_and_sort(
    df: pd.DataFrame,
    *value_cols: str,
    source_col: str = "Name",
) -> pd.DataFrame:

    df = df.copy()

    def clean_name(raw: str) -> str:
        raw = unicodedata.normalize("NFKC", str(raw)).replace("\xa0", " ").strip()

        # Suffixe (am Ende)
        raw = re.sub(r"\s*,\s*krfr\.?\s*Stadt\s*$", "", raw, flags=re.IGNORECASE)
        raw = re.sub(r"\s*,\s*kreisfreie\s*Stadt\s*$", "", raw, flags=re.IGNORECASE)
        raw = re.sub(r"\s*,\s*Stadt\s*$", "", raw, flags=re.IGNORECASE)
        raw = re.sub(r"\s*,\s*Kreis\s*$", "", raw, flags=re.IGNORECASE)

        # Präfixe (am Anfang)
        raw = re.sub(r"^(Stadt|Kreis)\s+", "", raw, flags=re.IGNORECASE)

        return raw.strip()

    df["Name"] = df[source_col].apply(clean_name)

    # Städteregion Aachen -> Aachen
    mask = df["Name"].str.contains("aachen", case=False, na=False)
    df.loc[mask, "Name"] = "Aachen"

    # Spaltenauswahl: Name + value_cols oder alles außer Rohspalte
    base_cols = ["Name"]
    if value_cols:
        cols = base_cols + list(value_cols)
    else:
        cols = base_cols + [c for c in df.columns if c not in base_cols + [source_col]]

    cols = [c for c in cols if c in df.columns]
    cleaned_df = df[cols].reset_index(drop=True)


    return cleaned_df

In [4]:
# CSV-Datei einlesen
df = pd.read_csv("../../data/raw/2024/22517-02i.csv", sep=";", encoding="latin1", skiprows=6)
# Quelle: https://www.landesdatenbank.nrw.de/
print(df.info())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 238 entries, 0 to 237
Data columns (total 7 columns):
 #   Column                                                                       Non-Null Count  Dtype 
---  ------                                                                       --------------  ----- 
 0   Unnamed: 0                                                                   236 non-null    object
 1   Unnamed: 1                                                                   232 non-null    object
 2   Unnamed: 2                                                                   232 non-null    object
 3   Insgesamt                                                                    234 non-null    object
 4   Insgesamt.1                                                                  234 non-null    object
 5   Eingliederungshilfe für seelisch behinderte junge Menschen § 35a SGB VIII    234 non-null    object
 6   Eingliederungshilfe für seelisch behinderte junge M

In [5]:
# nach Kreisen und kreisfreien Städten und in Anspruch genommenen (35a) Hilfen filtern
df = df.rename(columns={"Unnamed: 2": "Name"})


# große kreisangehörige Städte filtern, in dem bei den Kreisen, in denen sich eine große Kommune befindet, ein entsprechendes Flag gesetzt wird
grosse_kommunen = "Arnsberg, Bergheim, Bergisch Gladbach, Bocholt, Castrop-Rauxel, Detmold, Dinslaken, Dormagen, Dorsten, Düren, Gladbeck, Grevenbroich, Gütersloh, Herford, Herten, Iserlohn, Kerpen, Lippstadt, Lüdenscheid, Lünen, Marl, Minden, Moers, Neuss, Paderborn, Ratingen, Recklinghausen, Rheine, Siegen, Troisdorf, Unna, Velbert, Viersen, Wesel, Witten"

kommunen = {k.strip().lower() for k in grosse_kommunen.split(",")}
pattern = "|".join(map(re.escape, kommunen))
name_l = df["Name"].astype(str).str.lower()
mask_special = name_l.str.contains("jugendamt", na=False) & name_l.str.contains(pattern, na=False)
df.loc[mask_special, "kreis_code_tmp"] = df.loc[mask_special, "Unnamed: 1"].astype(str).str[2:7]
large_codes = set(df.loc[mask_special, "kreis_code_tmp"].dropna())



df = df.dropna(axis=1, how="all")
df = df.dropna(axis=0, how="all")

df = df[df["Name"].str.contains("Kreis|krfr. Stadt|Städteregion", case=False,na=False)]
df = df[~df["Name"].str.contains("Kreisjugendamt", case=False, na=False)]


# 3) Flag nur bei exakt passender Kreis-Zeile setzen
df["Kreis mit großer Gemeinde"] = (
    df["Unnamed: 1"].astype(str).isin(large_codes)
)



print(df)

    Unnamed: 0 Unnamed: 1                                             Name  \
4         2024      05111                          Düsseldorf, krfr. Stadt   
5         2024      05112                            Duisburg, krfr. Stadt   
6         2024      05113                               Essen, krfr. Stadt   
7         2024      05114                             Krefeld, krfr. Stadt   
8         2024      05116                     Mönchengladbach, krfr. Stadt   
9         2024      05117                 Mülheim an der Ruhr, krfr. Stadt   
10        2024      05119                          Oberhausen, krfr. Stadt   
11        2024      05120                           Remscheid, krfr. Stadt   
12        2024      05122                            Solingen, krfr. Stadt   
13        2024      05124                           Wuppertal, krfr. Stadt   
14        2024      05154                                     Kleve, Kreis   
21        2024      05158                                  Mettm

In [6]:
# Aachen ist mittlerweile Städteregion, daher wird alles außer Städteregion gelöscht

df["Name"] = df["Name"].str.strip()
df = df[df["Name"] != "Aachen, krfr. Stadt"]
df = df[df["Name"] != "Aachen, Kreis"]

In [7]:
# Typ (Kreis/kreisfreie Stadt/großer Kreis) extrahieren
kreisfreie_keywords = ["krfr. Stadt", "kreisfreie Stadt", "Stadt"]
kreis_keywords = ["Kreis", "kreis"]

def extract(raw: str, grosser_kreis: bool):
    raw = str(raw).strip()
    grosser_kreis = bool(grosser_kreis)

    if any(k in raw for k in kreisfreie_keywords):
        return raw, "Kreisfreie Stadt"

    if any(k in raw for k in kreis_keywords):
        return raw, "Großer Kreis" if grosser_kreis else "Kleiner Kreis"

    return raw, "Unbekannt"

df[["Name", "Typ 1"]] = df.apply(
    lambda r: pd.Series(extract(r["Name"], r["Kreis mit großer Gemeinde"])),
    axis=1
)

df["Typ 2"] = df["Typ 1"].replace(
    {"Großer Kreis": "Kreis", "Kleiner Kreis": "Kreis"})


print((df["Typ 1"] == "Unbekannt").sum())

0


In [10]:
# Strings parsen und anpassen
df = clean_and_sort(df, "Typ 1", "Typ 2")
df.loc[df["Name"] == "Aachen", "Typ 2"] = "Städteregion"
df.loc[df["Name"] == "Aachen", "Typ 1"] = "Städteregion"

In [11]:
print(df)

                          Name             Typ 1             Typ 2
0                   Düsseldorf  Kreisfreie Stadt  Kreisfreie Stadt
1                     Duisburg  Kreisfreie Stadt  Kreisfreie Stadt
2                        Essen  Kreisfreie Stadt  Kreisfreie Stadt
3                      Krefeld  Kreisfreie Stadt  Kreisfreie Stadt
4              Mönchengladbach  Kreisfreie Stadt  Kreisfreie Stadt
5          Mülheim an der Ruhr  Kreisfreie Stadt  Kreisfreie Stadt
6                   Oberhausen  Kreisfreie Stadt  Kreisfreie Stadt
7                    Remscheid  Kreisfreie Stadt  Kreisfreie Stadt
8                     Solingen  Kreisfreie Stadt  Kreisfreie Stadt
9                    Wuppertal  Kreisfreie Stadt  Kreisfreie Stadt
10                       Kleve     Kleiner Kreis             Kreis
11                    Mettmann      Großer Kreis             Kreis
12           Rhein-Kreis Neuss      Großer Kreis             Kreis
13                     Viersen      Großer Kreis             K

In [12]:
# saubere Tabelle abspeichern
df.to_csv("typen.csv", index=False)